# Egg Sex Classification — ResNetBiT 5-Fold Cross-Validation

Trains a pretrained ResNetBiT model (`resnetv2_50x1_bit.goog_in21k`) adapted for single-channel (grayscale) LSCI images. Uses 5-fold cross-validation with per-egg-ID prediction aggregation.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q timm torcheval torchmetrics torch_optimizer

In [ ]:
import os
import math
import random
import pickle
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFilter

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

import timm
from torcheval.metrics import BinaryAccuracy
from torchmetrics import AUROC
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix
from torch_optimizer import Lookahead

## 2. Configuration

All hyperparameters and paths are defined here for easy modification.

In [ ]:
# ── Reproducibility ──────────────────────────────────────────
RANDOM_SEED = 1839

# ── Model ────────────────────────────────────────────────────
MODEL_NAME  = 'resnetv2_50x1_bit.goog_in21k'
NUM_CLASSES = 1
PRETRAINED  = True

# ── Training ─────────────────────────────────────────────────
N_EPOCHS       = 120
LR             = 1e-4
BATCH_SIZE     = 64
BATCH_SIZE_VAL = 64
K_FOLDS        = 5

# ── Data ─────────────────────────────────────────────────────
DATASET_PATH = '/content/drive/MyDrive/CS 163/egg_sex_zoomed/full_256_cv/data'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 3. Utility Functions

In [ ]:
def set_seed(seed):
    """Set random seeds for reproducibility across Python, NumPy, and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def unsharp_mask(img, strength=0.95):
    """Sharpen a PIL image using an unsharp mask filter."""
    blurred    = img.filter(ImageFilter.GaussianBlur(radius=3.0))
    img_np     = np.array(img,     dtype=np.float32)
    blurred_np = np.array(blurred, dtype=np.float32)
    sharpened  = np.clip(img_np + strength * (img_np - blurred_np), 0, 255).astype(np.uint8)
    return Image.fromarray(sharpened)

## 4. Dataset Classes

In [ ]:
class CustomDataset(Dataset):
    """
    Loads all pickle files from a directory. Used to build the full dataset
    before splitting into cross-validation folds.
    """
    def __init__(self, pickle_dir):
        self.data = []
        self.ids  = []
        pickle_files = sorted(f for f in os.listdir(pickle_dir) if f.endswith('.pickle') and not f.startswith('.'))
        for fname in pickle_files:
            with open(os.path.join(pickle_dir, fname), 'rb') as f:
                self.data.extend(pickle.load(f))
        self.ids = [dp['additional_info']['id'] for dp in self.data]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        dp    = self.data[index]
        image = dp['image']
        np_image = np.array(image).astype(np.float32) / 255.0
        mean, std = np_image.mean(), np_image.std()
        np_image  = (np_image - mean) / (std + 1e-8)
        tensor = torch.tensor(np_image).unsqueeze(0).float()
        label  = torch.tensor(int(dp['label']), dtype=torch.float32)
        return tensor, label, dp['additional_info']


class SubsetDataset(Dataset):
    """
    Wraps a list of data-point dicts (a fold slice).
    Applies unsharp-mask sharpening to all images, and random affine
    augmentation during training (disabled when is_val=True).
    """
    def __init__(self, data, is_val=False):
        self.data = data
        self.ids  = [dp['additional_info']['id'] for dp in data]

        self.sharpen  = transforms.Lambda(unsharp_mask)
        self.augment  = transforms.Compose([]) if is_val else transforms.Compose([
            transforms.RandomAffine(
                degrees=180,
                translate=(0.05, 0.05),
                scale=(0.9, 1.1),
                fill=0,
            )
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        dp    = self.data[index]
        image = self.sharpen(dp['image'])

        np_image = np.array(image).astype(np.float32) / 255.0
        if len(self.augment.transforms) > 0:
            pil = Image.fromarray((np_image * 255).astype(np.uint8))
            np_image = np.array(self.augment(pil)).astype(np.float32) / 255.0

        # Per-image standardisation followed by ImageNet grayscale normalisation
        mean, std = np_image.mean(), np_image.std()
        np_image  = (np_image - mean) / (std + 1e-8)
        np_image  = (np_image - 0.449) / 0.226

        tensor = torch.tensor(np_image).unsqueeze(0).float()
        label  = torch.tensor(int(dp['label']), dtype=torch.float32)
        return tensor, label, dp['additional_info']

## 5. Model

In [ ]:
def create_grayscale_model(model_name, pretrained=True, num_classes=1, device=None):
    """
    Load a pretrained timm model and adapt its first convolutional layer to
    accept single-channel (grayscale) input by averaging the RGB weights.
    Currently supports ResNetBiT-style models that expose a `stem` attribute.
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)

    # Find the first Conv2d in the stem
    first_conv = None
    if hasattr(model, 'stem'):
        for layer in model.stem.modules():
            if isinstance(layer, nn.Conv2d) and layer.in_channels == 3:
                first_conv = layer
                break

    if first_conv is None:
        raise ValueError(f'No suitable first convolutional layer found for {model_name}.')

    new_conv = nn.Conv2d(
        in_channels=1,
        out_channels=first_conv.out_channels,
        kernel_size=first_conv.kernel_size,
        stride=first_conv.stride,
        padding=first_conv.padding,
        bias=first_conv.bias is not None,
    )
    with torch.no_grad():
        new_conv.weight[:] = first_conv.weight.mean(dim=1, keepdim=True)

    first_conv.in_channels = 1
    first_conv.weight      = new_conv.weight
    if first_conv.bias is not None:
        first_conv.bias = new_conv.bias

    return model

## 6. Cross-Validation Splits

In [ ]:
def cross_validation_splits(dataset, k_folds=5, random_seed=42):
    """
    K-fold split that keeps all images of the same egg ID in the same fold,
    preventing data leakage between training and validation sets.
    """
    unique_ids = list(set(dataset.ids))
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=random_seed)

    splits = []
    for train_idx, val_idx in kf.split(unique_ids):
        train_ids = {unique_ids[i] for i in train_idx}
        val_ids   = {unique_ids[i] for i in val_idx}
        train_data = [dp for dp in dataset.data if dp['additional_info']['id'] in train_ids]
        val_data   = [dp for dp in dataset.data if dp['additional_info']['id'] in val_ids]
        splits.append((train_data, val_data))

    return splits


set_seed(RANDOM_SEED)
dataset_full = CustomDataset(DATASET_PATH)
splits = cross_validation_splits(dataset_full, k_folds=K_FOLDS, random_seed=RANDOM_SEED)
print(f'Dataset size: {len(dataset_full)} images across {len(set(dataset_full.ids))} unique egg IDs')

## 7. Training & Validation Loops

Predictions are aggregated **per egg ID** before computing accuracy and AUC: eggs with an even number of images use mean probability; eggs with an odd number use majority voting. This reflects the real-world inference scenario.

In [ ]:
def train(dataloader, model, loss_fn, optimizer, device):
    """Train for one epoch; accuracy and AUC are computed per image."""
    model.train()
    running_loss     = 0.0
    accuracy_metric  = BinaryAccuracy(threshold=0.5)
    auc_metric       = AUROC(task='binary')

    for images, labels, _ in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        preds = model(images).sigmoid().squeeze(1)
        loss  = loss_fn(preds, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        accuracy_metric.update(preds, labels)
        auc_metric.update(preds, labels)

    epoch_loss = running_loss / len(dataloader)
    acc = accuracy_metric.compute().item()
    auc = auc_metric.compute().item()
    print(f'  Train  loss={epoch_loss:.4f}  acc={acc:.3f}  AUC={auc:.4f}')
    return epoch_loss, acc, auc


@torch.no_grad()
def validate(dataloader, model, loss_fn, device):
    """Validate for one epoch; accuracy and AUC are computed per image."""
    model.eval()
    running_loss    = 0.0
    accuracy_metric = BinaryAccuracy(threshold=0.5)
    auc_metric      = AUROC(task='binary')

    for images, labels, _ in dataloader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).sigmoid().squeeze(1)
        running_loss += loss_fn(preds, labels).item()
        accuracy_metric.update(preds, labels)
        auc_metric.update(preds, labels)

    epoch_loss = running_loss / len(dataloader)
    acc = accuracy_metric.compute().item()
    auc = auc_metric.compute().item()
    print(f'  Val    loss={epoch_loss:.4f}  acc={acc:.3f}  AUC={auc:.4f}')
    return epoch_loss, acc, auc


@torch.no_grad()
def validate_per_egg(dataloader, model, loss_fn, device):
    """
    Post-hoc validation where accuracy and AUC are aggregated per egg ID.
    Even number of images per egg: use mean probability.
    Odd number of images per egg: use majority voting.
    Used for final evaluation after training is complete.
    """
    model.eval()
    running_loss       = 0.0
    predictions_per_id = defaultdict(list)
    labels_per_id      = defaultdict(list)

    for images, labels, info in dataloader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).sigmoid().squeeze(1)
        running_loss += loss_fn(preds, labels).item()
        for i, eid in enumerate(info['id']):
            predictions_per_id[eid.item()].append(preds[i].item())
            labels_per_id[eid.item()].append(labels[i].item())

    agg_preds, agg_labels = [], []
    for eid in predictions_per_id:
        preds_list  = predictions_per_id[eid]
        labels_list = labels_per_id[eid]
        if len(preds_list) % 2 == 0:
            agg_preds.append(np.mean(preds_list))
        else:
            agg_preds.append(int(np.mean(np.round(preds_list)) > 0.5))
        agg_labels.append(np.mean(labels_list))

    agg_preds  = torch.tensor(agg_preds,  device=device)
    agg_labels = torch.tensor(agg_labels, device=device)
    acc_metric = BinaryAccuracy(threshold=0.5)
    auc_metric = AUROC(task='binary')
    acc_metric.update(agg_preds, agg_labels)
    auc_metric.update(agg_preds, agg_labels)
    acc = acc_metric.compute().item()
    auc = auc_metric.compute().item()
    epoch_loss = running_loss / len(dataloader)
    print(f'  Val (per-egg)  loss={epoch_loss:.4f}  acc={acc:.3f}  AUC={auc:.4f}')
    return epoch_loss, acc, auc


class EarlyStopping:
    """Stop training when the monitored metric does not improve for `patience` epochs."""
    def __init__(self, patience=10, delta=0.01, path='best_model.pth', monitor='accuracy'):
        self.patience   = patience
        self.delta      = delta
        self.path       = path
        self.monitor    = monitor
        self.counter    = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, metric, model):
        if self.best_score is None:
            self.best_score = metric
            torch.save(model.state_dict(), self.path)
        elif metric < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = metric
            torch.save(model.state_dict(), self.path)
            self.counter = 0

    def load_best(self, model):
        model.load_state_dict(torch.load(self.path))
        return model

## 8. Training Loop

In [ ]:
@torch.no_grad()
def get_confusion_matrix(dataloader, model, device):
    """Compute and return the confusion matrix for a given dataloader."""
    model.eval()
    all_preds, all_labels = [], []
    for images, labels, _ in dataloader:
        preds = model(images.to(device)).sigmoid().squeeze(1)
        all_preds.append(preds.cpu())
        all_labels.append(labels)
    return confusion_matrix(
        torch.cat(all_labels).numpy(),
        torch.cat(all_preds).numpy().round(),
    )


train_loss_all, train_acc_all, train_auc_all = [], [], []
val_loss_all,   val_acc_all,   val_auc_all   = [], [], []
trained_models = []

for fold, (train_data, val_data) in enumerate(splits):
    print(f'\n{"="*50}')
    print(f'Fold {fold + 1} / {K_FOLDS}')
    print(f'  Training samples:   {len(train_data)}')
    print(f'  Validation samples: {len(val_data)}')

    dataset_train = SubsetDataset(train_data, is_val=False)
    dataset_val   = SubsetDataset(val_data,   is_val=True)

    assert not set(dataset_val.ids) & set(dataset_train.ids), 'ID overlap between train and val!'

    dl_train = DataLoader(dataset_train, batch_size=BATCH_SIZE,     shuffle=True,  drop_last=True)
    dl_val   = DataLoader(dataset_val,   batch_size=BATCH_SIZE_VAL, shuffle=False, drop_last=False)

    model     = create_grayscale_model(MODEL_NAME, pretrained=PRETRAINED, num_classes=NUM_CLASSES, device=DEVICE).to(DEVICE)
    loss_fn   = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.999))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
    stopper   = EarlyStopping(patience=10, path=f'best_model_fold{fold+1}.pth', monitor='accuracy')

    fold_train_loss, fold_train_acc, fold_train_auc = [], [], []
    fold_val_loss,   fold_val_acc,   fold_val_auc   = [], [], []

    for epoch in range(N_EPOCHS):
        print(f'  Epoch {epoch+1}/{N_EPOCHS}')
        tl, ta, tu = train(dl_train, model, loss_fn, optimizer, DEVICE)
        vl, va, vu = validate(dl_val,   model, loss_fn, DEVICE)

        fold_train_loss.append(tl); fold_train_acc.append(ta); fold_train_auc.append(tu)
        fold_val_loss.append(vl);   fold_val_acc.append(va);   fold_val_auc.append(vu)

        scheduler.step()

        # Early stopping starts after a 40-epoch warm-up period
        if epoch >= 40:
            metric = vu if stopper.monitor == 'auc' else va
            stopper(metric, model)
            if stopper.early_stop:
                print(f'  Early stopping at epoch {epoch+1}.')
                stopper.load_best(model)
                break

    trained_models.append(model)
    train_loss_all.append(fold_train_loss); train_acc_all.append(fold_train_acc); train_auc_all.append(fold_train_auc)
    val_loss_all.append(fold_val_loss);     val_acc_all.append(fold_val_acc);     val_auc_all.append(fold_val_auc)

    cm = get_confusion_matrix(dl_val, model, DEVICE)
    print(f'\n  Confusion matrix (fold {fold+1}):\n{cm}')

## 9. Save Training Results

In [ ]:
results = {
    'train_loss': train_loss_all, 'val_loss': val_loss_all,
    'train_acc':  train_acc_all,  'val_acc':  val_acc_all,
    'train_auc':  train_auc_all,  'val_auc':  val_auc_all,
}
with open('resnetbit_training_results.pkl', 'wb') as f:
    pickle.dump(results, f)
print('Saved training results.')

## 10. Plot Training Curves

In [ ]:
# Load results (run this cell to reload from a saved file)
with open('/content/resnetbit_training_results.pkl', 'rb') as f:
    results = pickle.load(f)

train_loss_all = results['train_loss']
val_loss_all   = results['val_loss']
train_acc_all  = results['train_acc']
val_acc_all    = results['val_acc']
train_auc_all  = results['train_auc']
val_auc_all    = results['val_auc']


def smooth(data, window=5):
    """Apply a simple moving-average smoother."""
    smoothed = np.convolve(data, np.ones(window) / window, mode='valid')
    return np.append(smoothed, data[-1])


def plot_metric(train_data, val_data, ylabel, title, save_path):
    sns.set_theme(style='whitegrid')
    colors = sns.color_palette('tab10', K_FOLDS)
    plt.figure(figsize=(9, 6))
    for i, (td, vd) in enumerate(zip(train_data, val_data)):
        plt.plot(smooth(td), linestyle='dashed', color=colors[i], alpha=0.8, label=f'Train Fold {i+1}')
        plt.plot(smooth(vd), color=colors[i],    alpha=0.8,        label=f'Val Fold {i+1}')
    plt.xlabel('Epoch', fontsize=14)
    plt.ylabel(ylabel,  fontsize=14)
    plt.title(title,    fontsize=16)
    plt.legend(fontsize=10, frameon=True)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()


# Trim the last 10 epochs (often noisy near early-stopping boundary)
trim = lambda lst: [x[:-10] for x in lst]

plot_metric(trim(train_loss_all), trim(val_loss_all), 'BCE Loss', 'Training and Validation Loss',     'resnetbit_loss.png')
plot_metric(trim(train_acc_all),  trim(val_acc_all),  'Accuracy', 'Training and Validation Accuracy', 'resnetbit_acc.png')
plot_metric(trim(train_auc_all),  trim(val_auc_all),  'AUC',      'Training and Validation AUC',      'resnetbit_auc.png')

## 11. Confusion Matrices

In [ ]:
conf_matrices = [
    np.array([[260,  52], [159,  98]]),
    np.array([[185,  81], [151, 125]]),
    np.array([[172, 106], [105, 163]]),
    np.array([[166,  93], [106, 177]]),
    np.array([[215,  93], [140, 124]]),
]

fig, axes = plt.subplots(1, K_FOLDS, figsize=(14, 3))
for i, (cm, ax) in enumerate(zip(conf_matrices, axes)):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Male', 'Female'], yticklabels=['Male', 'Female'])
    ax.set_title(f'Fold {i+1}', fontsize=12)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Per-Fold Confusion Matrices', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('resnetbit_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

avg_cm = np.mean(conf_matrices, axis=0).astype(int)
plt.figure(figsize=(5, 4))
sns.heatmap(avg_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Male', 'Female'], yticklabels=['Male', 'Female'])
plt.title('Average Confusion Matrix (5 Folds)', fontsize=14)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('resnetbit_avg_confusion_matrix.png', dpi=150)
plt.show()

## 12. Save & Load Model Weights

In [ ]:
# Save all fold weights
for i, model in enumerate(trained_models):
    torch.save(model.state_dict(), f'resnetbit_fold{i+1}.pt')
    print(f'Saved resnetbit_fold{i+1}.pt')

In [ ]:
# Load a specific fold's weights for inference
fold_to_load = 1
model = create_grayscale_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES, device=DEVICE).to(DEVICE)
model.load_state_dict(torch.load(f'resnetbit_fold{fold_to_load}.pt', map_location=DEVICE))
model.eval()
print(f'Loaded fold {fold_to_load} weights.')